In [2]:
%pip install -U \
  "autogluon.timeseries==1.5.0" \
  "torch>=2.6.0" \
  "torchvision>=0.21.0" \
  "torchaudio>=2.6.0" \
  "transformers>=4.48.0" \
  "tokenizers>=0.21.0" \
  "numpy==1.26.4"

  Using cached torch-2.12.0-cp310-cp310-manylinux_2_28_x86_64.whl (532.1 MB)
  Using cached torchvision-0.27.0-cp310-cp310-manylinux_2_28_x86_64.whl (7.6 MB)
  Using cached transformers-5.8.1-py3-none-any.whl (10.6 MB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Using cached psutil-7.1.3-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl (263 kB)
  Using cached torchvision-0.26.0-cp310-cp310-manylinux_2_28_x86_64.whl (7.5 MB)
  Using cached torchvision-0.25.0-cp310-cp310-manylinux_2_28_x86_64.whl (8.1 MB)
  Using cached huggingface_hub-1.15.0-py3-none-any.whl (663 kB)
  Using cached typer-0.25.1-py3-none-any.whl (58 kB)
  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Using cached transformers-4.57.5-py3-none-any.whl (12.0 MB)
  Using cached transformers-4.57.4-py3-none-any.whl (12.0 MB)
  Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
  Using cached transformers-4.57.2-py3

In [1]:
import pandas as pd
import numpy as np
import findspark
import pyspark as spark
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField,FloatType,TimestampType,StringType
from pyspark.sql.functions import col,to_json,struct,from_json,to_timestamp,count, collect_list

In [2]:
pm10_predictor = TimeSeriesPredictor.load(path='../Offline-Phase/chronos2_model_pm10_bitola')
pm25_predictor = TimeSeriesPredictor.load(path = '../Offline-Phase/chronos2_model_pm25_bitola')

### Pandas Functions from the offline phase

In [3]:
def extract_time_features(df, timestamp_col='timestamp'):
    month = df[timestamp_col].dt.month

    df['season'] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11])
        ],
        [
            'winter',
            'spring',
            'summer',
            'autumn'
        ]
    )

    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    
    
    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)

    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

 
    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [4]:
def append_neighbors(df_hourly, neighbors_df, weather_cols=["humidity", "pressure","temperature", "wind_speed"], k_search=20, k_keep=3):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [5]:
neighbourhood_matrix = pd.read_csv("../data/neighbors_data/bitola_sensor_distances.csv")
neighbourhood_matrix

,sensor_id,neighbor_id,distance_km
0,d241a044-0a06-40c2-9d90-c91fd0a95060,fec52a19-9148-4350-a1b4-ae0da05ee199,7.082000
1,fec52a19-9148-4350-a1b4-ae0da05ee199,d241a044-0a06-40c2-9d90-c91fd0a95060,7.082000
2,d241a044-0a06-40c2-9d90-c91fd0a95060,be427cee-4c3a-4aa2-a1ce-9795a74533be,8.838588
3,be427cee-4c3a-4aa2-a1ce-9795a74533be,d241a044-0a06-40c2-9d90-c91fd0a95060,8.838588
4,d241a044-0a06-40c2-9d90-c91fd0a95060,c3f3da9b-9fd3-4037-94d3-598d655e6be9,10.004966
...,...,...,...
457,7b316592-8036-41e2-b8dc-b06b6a9afd54,40f081a6-4095-43f7-bffb-64e2af8c026e,1.049671
458,40f081a6-4095-43f7-bffb-64e2af8c026e,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,3.044465
459,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,40f081a6-4095-43f7-bffb-64e2af8c026e,3.044465
460,7b316592-8036-41e2-b8dc-b06b6a9afd54,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,2.083640


In [6]:
def load_context():
    context_df = pd.read_csv("context_bitola.csv")
    context_df['timestamp'] = pd.to_datetime(context_df['timestamp'],utc=True)
    return context_df

In [7]:
def load_forecast():
    forecast_df = pd.read_csv('../data/raw/bitola_forecast_weather.csv')
    forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'],utc=True)
    forecast_df = extract_time_features(forecast_df)
    forecast_df = append_neighbors(forecast_df, neighbourhood_matrix)
    
    return forecast_df

In [8]:
def build_future_df(ts_df, forecast_df):

    ts_df = ts_df.copy()
    forecast_df = forecast_df.copy()

    ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)
    forecast_df["timestamp"] = pd.to_datetime(forecast_df["timestamp"], utc=True)

    future_rows = []

    for sensor_id in ts_df["sensorId"].unique():

        sensor_current = ts_df[ts_df["sensorId"] == sensor_id]

        base_time = sensor_current["timestamp"].max()

        future_times = [
            base_time + pd.Timedelta(hours=i)
            for i in range(1, 513)
        ]

        sensor_future = pd.DataFrame({
            "sensorId": sensor_id,
            "timestamp": future_times
        })

        sensor_forecast = forecast_df[
            forecast_df["sensorId"] == sensor_id
        ]

        sensor_future = sensor_future.merge(
            sensor_forecast,
            on=["sensorId", "timestamp"],
            how="left"
        )

        future_rows.append(sensor_future)

    future_df = pd.concat(future_rows, ignore_index=True)

    return future_df

In [24]:
def predict_target(current_df, context_scaled, target, predictor, ID_COL, TIME_COL):

    context_target = context_scaled.copy()

    context_target[TIME_COL] = context_target[TIME_COL].dt.tz_convert(None)
    data = TimeSeriesDataFrame.from_data_frame(
        context_target,
        id_column=ID_COL,
        timestamp_column=TIME_COL
    )
    future_structure = predictor.make_future_data_frame(data)
    future_df_flat = future_structure.reset_index()
    
    # 3. Align your streaming batch's 24-hour data to timezone-naive
    current_df_cleaned = current_df.copy()
    current_df_cleaned[TIME_COL] = current_df_cleaned[TIME_COL].dt.tz_convert(None)
    
    # 4. Merge your 24-hour real data onto the 512-step structural skeleton
    padded_covariates = future_df_flat.merge(
        current_df_cleaned,
        left_on=["item_id", "timestamp"],
        right_on=[ID_COL, TIME_COL],
        how="left"
    )
    
    # Preserve the structural identity column names
    if ID_COL in padded_covariates.columns and ID_COL != "item_id":
        padded_covariates[ID_COL] = padded_covariates["item_id"]

    # 5. Stretch your 24 hours of weather data to cover all 512 steps via forward-fill
    covariate_cols = ["humidity", "pressure", "temperature", "wind_speed"]
    for col_name in covariate_cols:
        if col_name in padded_covariates.columns:
            padded_covariates[col_name] = padded_covariates.groupby("item_id")[col_name].ffill().bfill()
            
    # 6. Convert final padded covariate frame back into an AutoGluon format
    known_covariates_ts = TimeSeriesDataFrame.from_data_frame(
        padded_covariates,
        id_column="item_id",
        timestamp_column="timestamp"
    )

    # 7. Execute full prediction safely
    predictions = predictor.predict(
        data=data,
        known_covariates=known_covariates_ts
    )

    pred_df = predictions.to_data_frame()

    
    if isinstance(pred_df.index, pd.MultiIndex):
        pred_df = pred_df.reset_index()

    pred_df = pred_df.rename(columns={
        "item_id": ID_COL,
        "timestamp": TIME_COL
    })
    pred_df[TIME_COL] = pd.to_datetime(pred_df[TIME_COL],utc=True)
    pred_df = pred_df.sort_values([ID_COL, TIME_COL])

  
    current_pred = (
        pred_df
        .groupby(ID_COL)
        .nth(0)  
        .reset_index()
    )

    current_pred = current_pred.rename(columns={"mean": target})

    forecast_24h = (
        pred_df
        .groupby(ID_COL)["mean"]
        .apply(lambda x: x.head(24).tolist())
        .reset_index()
        .rename(columns={"mean": f"{target}_forecast_24h"})
    )

    result_df = current_df.merge(
        current_pred[[ID_COL, target]],
        on=ID_COL,
        how="left"
    )

    result_df = result_df.merge(
        forecast_24h,
        on=ID_COL,
        how="left"
    )

    return result_df

In [10]:
def process_batch(current_df, context_df):
    ID_COL = "sensorId"
    TIME_COL = "timestamp"

    numeric_features = [
        'humidity', 'pressure', 'temperature', 'wind_speed',
        'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
        'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
        'neighbor1_temperature', 'neighbor2_temperature', 'neighbor3_temperature',
        'neighbor1_wind_speed', 'neighbor2_wind_speed', 'neighbor3_wind_speed'
    ]

    current_df[TIME_COL] = pd.to_datetime(current_df[TIME_COL], utc=True)
    context_df[TIME_COL] = pd.to_datetime(context_df[TIME_COL], utc=True)
    

    # go pravime ova za da osigurame deka i pdf i context_df imaat site potrebni koloni, ako ne, da gi dodademe so NaN vrednosti
    # za da ne crashne modelot posle
    for col in numeric_features:
        if col not in current_df.columns:
            current_df[col] = np.nan
        if col not in context_df.columns:
            context_df[col] = np.nan

    context_scaled = context_df.copy().sort_values([ID_COL, TIME_COL])

    print("Context max timestamp:", context_scaled[TIME_COL].max())

    

    pm10_df = predict_target(
    current_df, context_scaled,
    target="pm10",
    predictor= pm10_predictor,
    ID_COL=ID_COL,
    TIME_COL=TIME_COL
    )

    pm25_df = predict_target(
    current_df, context_scaled,
    target="pm25",
    predictor= pm25_predictor,
    ID_COL=ID_COL,
    TIME_COL=TIME_COL
    )

    return pm10_df, pm25_df

In [11]:
def write_to_kafka(df, topic):
    spark_df = spark.createDataFrame(df)

    kafka_df = spark_df.select(
        col("sensorId").cast("string").alias("key"),
        to_json(struct(*spark_df.columns)).alias("value")
    )

    kafka_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", topic) \
        .save()

In [12]:
context_df = load_context()
forecast_df = load_forecast()
def foreach_batch(batch_df, epoch_id):
    global context_df

    print(f"\nBatch received! Epoch: {epoch_id}")

    if batch_df.rdd.isEmpty():
        return

    current_df= batch_df.toPandas()

    if current_df.empty:
        print("Empty batch — skipping")
        return

    current_df = current_df.sort_values("timestamp")

    for _, row in current_df.iterrows():
        ts = row["timestamp"]

        ts_df = pd.DataFrame([r.asDict() for r in row["rows"]])

        print(f"\nProcessing timestamp: {ts}")
        print(f"Sensor count: {len(ts_df)}")

        context_max_ts = pd.to_datetime(context_df["timestamp"], utc=True).max() if not context_df.empty else None
        if context_max_ts is not None and pd.to_datetime(ts, utc=True) <= context_max_ts:
            print(f"Skipping timestamp {ts} because it is not after context max {context_max_ts}")
            continue
        
        incoming_ids = set(ts_df['sensorId'].unique())
        existing_ids = set(context_df['sensorId'].unique()) if not context_df.empty else set()
        new_ids = incoming_ids - existing_ids

        if new_ids:
            print(f"New sensors detected: {new_ids}")

            numeric_columns = [
                col for col in context_df.columns
                if col in ['temperature','wind_speed','humidity','pm10','pm25','pressure']
            ]

            city_baseline = context_df.groupby('timestamp')[numeric_columns].median().reset_index()

            proxy_rows = []

            for sid in new_ids:
                proxy_history = city_baseline.copy()
                proxy_history['sensorId'] = sid

                temp_combined = pd.concat([context_df, proxy_history], ignore_index=True)

                refined_data = append_neighbors(temp_combined, neighbourhood_matrix)
                refined_data = extract_time_features(refined_data)

                new_sensor_proxy = refined_data[refined_data['sensorId'] == sid]
                proxy_rows.append(new_sensor_proxy)

            context_df = pd.concat([context_df, *proxy_rows], ignore_index=True)

        ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)

        ts_df = extract_time_features(ts_df)
        ts_df = append_neighbors(ts_df, neighbourhood_matrix)

        future_df = build_future_df(ts_df,forecast_df)

        pm10_df, pm25_df = process_batch(future_df, context_df)
     

        if pm10_df is None or pm25_df is None:
            print("Prediction skipped")
            continue

        write_to_kafka(pm10_df, topic="FullPm10WeatherData")
        write_to_kafka(pm25_df, topic="FullPm25WeatherData")
        
        ts_df = ts_df.merge(
            pm10_df[['sensorId', 'timestamp', 'pm10']],
            on=['sensorId', 'timestamp'],
            how='left'
        )

        ts_df = ts_df.merge(
            pm25_df[['sensorId', 'timestamp', 'pm25']],
            on=['sensorId', 'timestamp'],
            how='left'
        )

        context_df = pd.concat([context_df, ts_df], ignore_index=True)

        context_df = (
            context_df
            .sort_values(["sensorId", "timestamp"])
            .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
            .groupby("sensorId")
            .tail(72)
            .reset_index(drop=True)
        )

        print(f"Prediction done for {ts}")
        print(f"Context size: {len(context_df)}")

# Online Phase (Main Program)

In [13]:
findspark.init()

In [14]:
spark = SparkSession.builder \
    .appName("KafkaConsumerExample") \
    .config( "spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7")  \
    .getOrCreate()

:: loading settings :: url = jar:file:/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/jovan/.ivy2/cache
The jars for the packages stored in: /Users/jovan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-18ec21ba-7e9e-4129-bbb8-716a196df941;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 153ms :: artifacts dl 4ms
	:: 

In [15]:
print(spark.version)

3.5.7


In [16]:
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribePattern", "sensor_.*") \
    .load()

In [17]:
df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [18]:
schema = StructType([
    StructField("timestamp",TimestampType(),True),
    StructField("sensorId",StringType(),True),
    StructField("lat",FloatType(),True),
    StructField("lon",FloatType(),True),
    StructField("humidity",FloatType(),True),
    StructField("pressure",FloatType(),True),
    StructField("temperature",FloatType(),True),
    StructField("wind_speed",FloatType(),True)
])

In [19]:
parsed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .drop("lat", "lon")

In [20]:
parsed_df = parsed_df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [21]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- humidity: float (nullable = true)
 |-- pressure: float (nullable = true)
 |-- temperature: float (nullable = true)
 |-- wind_speed: float (nullable = true)



In [22]:
grouped_df = parsed_df \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy("timestamp") \
    .agg(
        collect_list(struct("*")).alias("rows"),
        count("*").alias("sensor_count")
    )

In [25]:
query = grouped_df.writeStream \
    .foreachBatch(foreach_batch) \
    .start()

query.awaitTermination()

26/07/13 21:33:18 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/temporary-f87211d7-0b02-4e2a-a7a6-3a065d4d70e1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/13 21:33:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/13 21:33:18 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.



Batch received! Epoch: 0



Batch received! Epoch: 1



Batch received! Epoch: 2



Batch received! Epoch: 3


...

data with frequency 'IRREG' has been resampled to frequency 'h'.
data with frequency 'IRREG' has been resampled to frequency 'h'.
data with frequency 'IRREG' has been resampled to frequency 'h'.
data with frequency 'IRREG' has been resampled to frequency 'h'.


Prediction done for 2025-12-01 08:00:00
Context size: 1584

Processing timestamp: 2025-12-01 09:00:00
Sensor count: 22
Context max timestamp: 2025-12-01 08:00:00+00:00


data with frequency 'IRREG' has been resampled to frequency 'h'.
data with frequency 'IRREG' has been resampled to frequency 'h'.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

26/07/13 21:36:36 ERROR Executor: Exception in task 8.0 in stage 102.0 (TID 2335)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1094, in main
    split_index = read_int(infile)
                  ^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 594, in read_int
    length = stream.read(4)
             ^^^^^^^^^^^^^^
KeyboardInterrupt

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at o